In [ ]:
from geo_data import data_handler

In [ ]:
deck_name = "Allgemeinwissen::01 🟦 Geografie 🌍::01.01 Ultimate Geography📍"
df = data_handler.anki_to_df(deck_name)
df.head()

# Other options
In the end, I created the function of loading by using the Python-module "anki" and reading in a copy of the collection database with this module. Before, I tried different approaches, which all did not really work; however, I liked what I learned and want to keep them her.

## .apkg-file
In Anki, decks can be exported as .apkg file. Under the hood, this is just a zip-file, containing a folder with the media (images etc.) called *collection.media* and a sqlite database called *collection.anki2*. The database can be read using SQL. Unfortunately, it was often weird to read this file in, because there was no data in it - I do not understand why. In the collection.anki2 of the local collection, it worked, but not for the .apkg...

### sqlite3 in console
In console, sqlite3 can be used. It can be installed with `sudo apt install sqlite3`. Then, it can be used to open the database by calling: `sqlite3 collection.anki2`. Afterwards, SQL can be used for access. Here are some examples:
```SQL
.quit
-- exit sqlite3
.tables
-- shows all tables in the database
PRAGMA table_info(notes);
-- shows the columns of the table "notes"
```

I suspect that the information about field names in note types is saved somewhere here:
```SQL
SELECT config FROM notetypes;
```
However, the data type of this column is `BLOB` and probably a format that is only readable in the backend of anki, hence it is hard for me in Python to get the field and names. But it is possible that I just did not find the correct place in the database where note types and field names are stored.

### sqlite3 in python
There is a python module in the standard library to load SQLite databases.

In [ ]:
import sqlite3

from geo_data.data.config import settings

In [ ]:
# first tries
path = settings.data_dir / "collection.anki2"
deck_name = "Guitar"

with sqlite3.connect(f"file:{path}?mode=ro", uri=True) as conn:
    cursor = conn.cursor()

    cursor.execute(f"SELECT id FROM decks WHERE name LIKE '%{deck_name}%';")
    deck_ids = [row[0] for row in cursor.fetchall()]

    deck_id = deck_ids[0]
    cursor.execute(f"SELECT DISTINCT(nid) FROM cards WHERE did = {deck_id};")
    note_ids = [row[0] for row in cursor.fetchall()]

    cmd = f"SELECT * FROM notes WHERE id in {note_ids};".replace("[", "(").replace(
        "]", ")"
    )
    cursor.execute(cmd)
    result = cursor.fetchall()

In [ ]:
# this would be much cleaner and easier with a single SQL command using JOIN
path = settings.data_dir / "collection.anki2"
deck_name = "Guitar"

with sqlite3.connect(f"file:{path}?mode=ro", uri=True) as conn:
    cursor = conn.cursor()
    query = """
        SELECT DISTINCT n.id, n.flds, n.tags, n.sfld
        FROM notes AS n
        JOIN cards AS c ON n.id = c.nid
        JOIN decks AS d ON c.did = d.id
        WHERE d.name LIKE ?
    """
    cursor.execute(query, (f"%{deck_name}%",))
    rows = cursor.fetchall()

    notes = [
        {
            "id": note_id,
            "fields": flds.split("\x1f"),
            "tags": tags.strip(),
            "sort_field": sfld,
        }
        for (note_id, flds, tags, sfld) in rows
    ]
notes

I also played around a little with the apkg-files and automatically unzip them, but this did not really work and I did not try to investigate it much further.

In [ ]:
import tempfile
import zipfile

In [ ]:
path = settings.data_dir / "Allgemeinwissen.apkg"

zf = zipfile.ZipFile(path)

with tempfile.TemporaryDirectory() as tempdir:
    zf.extractall(tempdir)